# Paré Regeneration Transcriptomic Analysis & Export Pipeline
**Project:** Paré et al. 2023 Replication & Sub-Categorical Filtering Workspace
**Authors:** Alina Krawczynski, Rannyele Passos Ribeiro (powered by Claude)
**Lab:** Özpolat Lab, Washington University in St. Louis, 1 Brookings Drive, St. Louis, MO 63130
**Year:** 2026

---

## Dataset Overview & Biological Context
This dashboard explores the **TMM-normalized expression matrix** for *Platynereis dumerilii* posterior
regeneration (Paré et al. 2023, BMC Genomics 24:583, `doi:10.1186/s12864-023-09602-z`) across six
regeneration stages: **0dpa → 1dpa → 2dpa → 3dpa → 5dpa** (days post-amputation) plus **NA**
(non-amputated control). It reads the already-cleaned tables produced by `Data Cleaning Analysis.ipynb`
(in `cleaned/`) rather than re-parsing the raw supplementary Excel files, so run that notebook first if
`cleaned/` is missing or you have re-downloaded the source files.

### Understanding the Expression Values
* **Generation:** Reads from 15 high-quality samples (2-3 replicates per stage, 3 outlier replicates were
  already excluded by the authors) were mapped to the de novo Reference transcriptome and TMM-normalized.
* **Normalization:** TMM (trimmed mean of M-values) rescales each sample's library so a handful of very
  highly-expressed transcripts cannot skew the comparison between samples. In plain terms, it corrects for
  differences in sequencing depth and RNA composition between samples, so a stage with more total reads
  does not just look like it has "more of everything."
* **Interpretation:** Each bar below is the **mean of that stage's replicates**, showing relative transcript
  abundance across the course of regeneration.

### Automated Data Cleaning & QC Rules Applied (recap from the cleaning notebook):
1. Gene IDs and sample columns are standardized to strings. Outlier replicates are already excluded upstream.
2. Trinotate's `.` missing-value placeholder was converted to real `NaN`, and transcript-level annotation was
   collapsed to one row per gene (preferring an annotated isoform when one exists).
3. The curated neurogenesis/stem-cell marker list's 3-row header and its `"NA"` stage label (which pandas'
   default NA-detection otherwise silently swallows) were parsed explicitly by column position.

---

## Key Features
* **Hybrid Gene Input:** Search transcripts dynamically using a live-filtering list (hold `Ctrl`/`Cmd` to
  multi-select), paste a raw block of `PladumEVm...` IDs, or search by annotation keyword.
* **Flexible Stage Filtering:** Isolate a single regeneration stage, or hold `Ctrl`/`Cmd` to select any custom
  combination of stages.
* **Dual-Source Annotation:** Each searched gene is checked against the curated **neurogenesis/stem-cell
  marker list** (gene symbol) and the **Trinotate BLASTX annotation** (functional description), matching on
  the shared `PladumEVm...` gene ID.
* **Sequential DEG Status:** For each gene, shows whether it was significantly up/down-regulated in each of
  the 5 sequential stage comparisons (0 vs 1dpa, 1 vs 2dpa, 2 vs 3dpa, 3 vs 5dpa, 5dpa vs NA), using the
  paper's own precomputed calls for curated marker genes, and the DEG results table for everything else.
* **On-Demand Exporter:** A dual-button layout separates quick visualization from storage, with a
  customizable save-location (type a path, or use the folder-browse button, which defaults to `exported_results/`).
* **Multi-Gene Heatmap:** Selecting 2+ genes automatically adds a genes-by-stage, hierarchically clustered
  heatmap (row-normalized so genes at different expression scales stay visually comparable) with a
  colorblind-safe palette toggle, after the individual bar charts, included in the PDF export.

---

## Dashboard User Guide

### Step 1: Choose Regeneration Stages
Hold `Ctrl` (or `Cmd` on Mac) in the **Stages** box to highlight any custom combination. All stages are
selected by default.

### Step 2: Choose Target Transcripts (Use A, B, or C!)
* **Option A (Filter List):** Type into **Filter List** to shrink the selection panel in real-time, then
  click (or Ctrl/Cmd-click) transcript IDs.
* **Option B (Paste Box):** Drop a raw text block of `PladumEVm...` IDs copied from a spreadsheet straight
  into the box.
* **Option C (Search by Keyword):** type a keyword to find every gene whose curated marker symbol or
  Trinotate BLASTX description contains it.

### Step 3: Run and/or Save
* Click **🎯 Generate Visuals (View Only)** to render bar charts, the multi-gene heatmap, annotation
  tables, and DEG status tables instantly on-screen.
* Click **💾 Export Active Results (Save CSV/PDF)** to save high-resolution vector assets to your chosen
  folder (defaults to `exported_results/`).


In [ ]:
# ============================================================================================
#  DEPENDENCY CHECK. Run this cell first.
#  Installs any Python packages this notebook needs that are not already present, then you are
#  done. Safe to re-run (does nothing if everything is already installed). To install by hand
#  instead, run this in a terminal / Anaconda Prompt:
#      pip install pandas numpy matplotlib seaborn scipy ipywidgets
# ============================================================================================
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy', 'ipywidgets']

_missing = [p for p in REQUIRED_PACKAGES if importlib.util.find_spec(p) is None]
if _missing:
    print("Installing missing packages:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *_missing])
    print("\nInstall finished. If an import fails in a later cell, restart the kernel")
    print("(menu: Kernel -> Restart Kernel) and run the notebook again from the top.")
else:
    print("All required packages are already installed. No action is needed.")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# This packaged copy keeps all input files flat alongside the notebook (no cleaned/ subfolder,
# that layout was specific to the working repo, where this data was produced by a separate
# upstream cleaning notebook).

# 1. Load the master TMM expression matrix
df_tmm = pd.read_csv("tmm_expression_matrix.csv", index_col="gene_id", low_memory=False)
df_tmm.index = df_tmm.index.astype(str)

# 2. Load sample metadata (maps each sample column to its regeneration stage + replicate number).
# keep_default_na=False is required here: the "NA" stage label (non-amputated control) is a real
# category value, not a missing value, but pandas' default NA-detection silently converts a bare
# "NA" cell to an actual NaN otherwise, which would make the NA stage match zero sample columns.
df_sample_metadata = pd.read_csv("sample_metadata.csv", index_col="sample", keep_default_na=False)

# 3. Load annotation sources. gene_annotation_trinotate_trimmed.csv.gz is a trimmed copy of the
# full Trinotate report. The original carries 16 columns (transcript IDs, RNAMMER, Pfam,
# SignalP, TmHMM, eggnog, Kegg, multiple GO-term columns, etc.), of which this dashboard only ever
# reads gene_id and blastx_description. Trimming + gzip-compressing shrinks it from 120MB to under
# 1MB with byte-identical values for the columns actually used. This is what keeps the whole
# dashboard package small enough to host on GitHub.
df_annotation = pd.read_csv("gene_annotation_trinotate_trimmed.csv.gz", index_col="gene_id", low_memory=False)
df_annotation.index = df_annotation.index.astype(str)

df_markers = pd.read_csv("curated_marker_genes.csv")
df_markers["transcriptome_id"] = df_markers["transcriptome_id"].astype(str)

df_deg = pd.read_csv("deg_all_comparisons.csv.gz", low_memory=False)
df_deg["gene_id"] = df_deg["gene_id"].astype(str)

# 4. Regeneration stages in their natural chronological order
STAGE_ORDER = ["0dpa", "1dpa", "2dpa", "3dpa", "5dpa", "NA"]

# The 5 "sequential" comparisons the paper treats as most biologically meaningful
SEQUENTIAL_COMPARISONS = [
    ("0dpavs1dpa", "0dpa vs 1dpa"),
    ("1dpavs2dpa", "1dpa vs 2dpa"),
    ("2dpavs3dpa", "2dpa vs 3dpa"),
    ("3dpavs5dpa", "3dpa vs 5dpa"),
    ("5dpavsNA", "5dpa vs NA"),
]

# Pre-index the (comparison, gene) -> logFC lookup once, restricted to the sequential comparisons,
# so per-gene DEG status lookups in the dashboard are fast instead of re-scanning 150k+ rows each time.
_seq_keys = [k for k, _ in SEQUENTIAL_COMPARISONS]
df_deg_seq = df_deg[df_deg["comparison"].isin(_seq_keys)]
_deg_lookup = {(row.comparison, row.gene_id): row.logFC for row in df_deg_seq.itertuples()}

# 5. Collapse the replicates into a single mean value per regeneration stage
df_mean = pd.DataFrame(index=df_tmm.index)
for stage in STAGE_ORDER:
    stage_cols = df_sample_metadata.index[df_sample_metadata["stage"] == stage].tolist()
    df_mean[stage] = df_tmm[stage_cols].mean(axis=1)

print(f"✅ Master Matrix Ready. {len(df_mean)} genes across {len(STAGE_ORDER)} regeneration stages (replicate-averaged).")


---

## 🔬 BLAST Sequence Search
Have a nucleotide sequence from elsewhere (a primer, a gene from another organism, a sequence copied
from a paper) and want to know if/where it shows up in this dataset? Paste it below to BLAST it
against Paré et al.'s own assembled transcriptome (a local `blastn` search, no internet required).
Any matching transcript ID(s) found can then be copied into the **Paste Box** in Step 2 above to view
that gene's expression profile. Click **💾 Export Hit Sequences** to save the hit table (CSV) and the
matched sequences plus your original query (FASTA) to `exported_results/`.

**Setup (optional, the rest of the dashboard works without this):** the sequence search needs
the free NCBI BLAST+ toolkit plus a small search database you build once from the reference
transcriptome. Step-by-step instructions (install BLAST+, get the reference FASTA, and build
the database into a `blast_db/` folder next to this notebook) are in the comment at the top of
the next code cell. The notebook finds BLAST+ automatically once it is installed. Until then this
section just prints a short reminder, and everything else in the notebook is unaffected.

In [ ]:
import shutil
import subprocess
import tempfile
import re

# --- Sequence -> Gene ID search: paste any raw nucleotide sequence and find the best-matching
# transcript(s) in Pare's own assembled transcriptome via a local blastn search. Useful when you
# have a sequence from elsewhere and want to know its gene ID in this dataset (and thus its
# expression profile via the dashboard above). Matched sequences can also be exported directly
# from the database itself. See export_blast_hits() below.
#
# SETUP (only needed once per machine): install BLAST+ from
# https://ftp.ncbi.nlm.nih.gov/blast/executables/blast+/ , then rebuild the database from the
# original transcriptome FASTA in two steps. First, rewrite each header so the PladumEVm...t#
# transcript ID is the first token (required for ID-based lookup/export, since the original
# "ENA|...|..." header style is not a BLAST-recognized accession format):
#   python -c "
#   import re
#   with open('assmebled_transcriptome_Parre_etal_2023/HBZZ01 copy.fasta') as fin, open('renamed.fasta', 'w') as fout:
#       for line in fin:
#           if line.startswith('>'):
#               m = re.search(r'(PladumEVm\d+t\d+)', line)
#               fout.write(f'>{m.group(1)} {line[1:].strip()}\n' if m else line)
#           else:
#               fout.write(line)
#   "
# Then build the database with -parse_seqids so sequences can be retrieved by that ID:
#   makeblastdb -in renamed.fasta -dbtype nucl -parse_seqids -out blast_db/pare_transcriptome
# The notebook auto-detects BLAST+ on your PATH (or set BLAST_BIN_DIR
# below) and looks for the database in ./blast_db/.
# Once built, the database is fully self-contained. The original transcriptome FASTA is not
# needed for searching OR sequence export, and can be deleted to save space if desired.

# --- BLAST+ location (auto-detected) ---------------------------------------------------------
# By default this notebook finds the BLAST+ programs (blastn, blastdbcmd, makeblastdb) on your
# system PATH. If you installed BLAST+ the normal way, leave BLAST_BIN_DIR = "" and it just
# works. If BLAST+ lives somewhere not on your PATH, set BLAST_BIN_DIR to the folder that holds
# the blastn / blastdbcmd / makeblastdb executables.
BLAST_BIN_DIR = ""   # e.g. r"C:\Program Files\NCBI\blast-2.17.0+\bin"  or  "/usr/local/ncbi/blast/bin"

# --- Optional automatic installation of BLAST+ (see the SETUP note above for the manual method)
# If BLAST+ is not found anywhere above, this cell offers a button that downloads the matching
# prebuilt NCBI BLAST+ archive for this operating system and CPU architecture, directly from
# https://ftp.ncbi.nlm.nih.gov/blast/executables/blast+/LATEST/ (approximately 150 to 250MB,
# downloaded once). The archive is verified against NCBI's published MD5 checksum, and only the
# 3 executables this notebook requires (blastn, blastdbcmd, makeblastdb) are extracted, into
# ./blast_bin/. No conda or other package manager is required (most users of this notebook are
# unlikely to have one installed already), and nothing is downloaded unless the button is
# clicked.
import platform
import urllib.request
import urllib.error
import hashlib
import tarfile
import stat

def _urlopen(url, timeout):
    """Opens a URL, working around a common macOS gap in which the python.org installer's own
    OpenSSL CA bundle is never populated (because the separate "Install Certificates.command"
    step was skipped). If the default SSL context cannot validate a normal HTTPS site, the
    request is retried using the certifi package's CA bundle."""
    try:
        return urllib.request.urlopen(url, timeout=timeout)
    except urllib.error.URLError as e:
        if "certificate verify failed" not in str(e.reason):
            raise
        import certifi
        import ssl
        ctx = ssl.create_default_context(cafile=certifi.where())
        return urllib.request.urlopen(url, timeout=timeout, context=ctx)


def _blast_archive_info():
    """Resolves which prebuilt NCBI BLAST+ archive matches this machine, and its exact current
    filename from NCBI's LATEST/ index page (the filename embeds a version number, e.g.
    ncbi-blast-2.17.0+-x64-macosx.tar.gz, so it cannot be hardcoded here). This approach
    continues to work as NCBI releases new versions, without requiring this notebook to be
    updated."""
    system = platform.system()
    machine = platform.machine().lower()
    if system == "Darwin":
        arch_tag = "aarch64-macosx" if machine in ("arm64", "aarch64") else "x64-macosx"
    elif system == "Linux":
        arch_tag = "aarch64-linux" if machine in ("aarch64", "arm64") else "x64-linux"
    elif system == "Windows":
        arch_tag = "x64-win64"
    else:
        return None

    index_url = "https://ftp.ncbi.nlm.nih.gov/blast/executables/blast+/LATEST/"
    try:
        with _urlopen(index_url, timeout=30) as resp:
            html = resp.read().decode("utf-8", errors="ignore")
    except Exception:
        return None

    match = re.search(r"ncbi-blast-[\d.]+\+-" + re.escape(arch_tag) + r"\.tar\.gz", html)
    if not match:
        return None
    filename = match.group(0)
    size_match = re.search(re.escape(filename) + r"</a>\s*\S+\s+\S+\s+([\d.]+[MKG]?)", html)
    return {
        "url": index_url + filename,
        "md5_url": index_url + filename + ".md5",
        "filename": filename,
        "size_label": size_match.group(1) if size_match else "~150-250MB",
        "exe_suffix": ".exe" if system == "Windows" else "",
    }


def install_blast_locally():
    """Downloads the matching prebuilt NCBI BLAST+ archive directly from NCBI, verifies it
    against NCBI's published MD5 checksum, and extracts blastn, blastdbcmd, and makeblastdb into
    ./blast_bin/ (re-downloading is skipped if these are already present). This function runs
    automatically when the install button shown by this cell is clicked, if BLAST+ is not found.
    It can also be called directly, as install_blast_locally(), to bypass the button."""
    local_dir = "blast_bin"
    wanted = {"blastn", "blastdbcmd", "makeblastdb"}
    exe_suffix = ".exe" if platform.system() == "Windows" else ""

    if all(os.path.isfile(os.path.join(local_dir, name + exe_suffix)) for name in wanted):
        print(f"✅ BLAST+ is already installed in ./{local_dir}/, no action needed.")
        return

    info = _blast_archive_info()
    if info is None:
        print(f"❌ No prebuilt BLAST+ archive is available for {platform.system()} {platform.machine()} "
              f"(or NCBI could not be reached). Install manually, following the SETUP note above this cell.")
        return

    os.makedirs(local_dir, exist_ok=True)
    archive_path = os.path.join(local_dir, info["filename"])
    print(f"⬇️ Downloading {info['filename']} ({info['size_label']}) from NCBI, usually 1 to 3 minutes...")

    extracted = set()
    try:
        downloaded, next_report_mb = 0, 20
        with _urlopen(info["url"], timeout=60) as resp, open(archive_path, "wb") as out:
            while True:
                chunk = resp.read(1024 * 1024)
                if not chunk:
                    break
                out.write(chunk)
                downloaded += len(chunk)
                if downloaded // (1024 * 1024) >= next_report_mb:
                    print(f"   ...{downloaded // (1024 * 1024)} MB downloaded")
                    next_report_mb += 20

        md5_expected = None
        try:
            with _urlopen(info["md5_url"], timeout=15) as resp:
                md5_expected = resp.read().decode().split()[0]
        except Exception:
            pass  # optional integrity check, proceeds without it if NCBI's .md5 file is unreachable

        if md5_expected:
            hasher = hashlib.md5()
            with open(archive_path, "rb") as f:
                for chunk in iter(lambda: f.read(1024 * 1024), b""):
                    hasher.update(chunk)
            if hasher.hexdigest() != md5_expected:
                print("❌ The downloaded file's checksum does not match NCBI's published MD5. The download")
                print("   may be corrupted. Re-run this cell, or click the button again, to retry.")
                return

        with tarfile.open(archive_path, mode="r:gz") as tar:
            for member in tar.getmembers():
                base = os.path.basename(member.name)
                if os.path.basename(os.path.dirname(member.name)) == "bin" and base in {w + exe_suffix for w in wanted}:
                    tar.extract(member, path=local_dir)
                    final_path = os.path.join(local_dir, base)
                    shutil.move(os.path.join(local_dir, member.name), final_path)
                    os.chmod(final_path, os.stat(final_path).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
                    extracted.add(base[: len(base) - len(exe_suffix)] if exe_suffix else base)
    except Exception as e:
        print(f"❌ Install failed: {e}")
        print("   BLAST+ can still be installed manually, following the SETUP note above this cell.")
        return
    finally:
        if os.path.isfile(archive_path):
            os.remove(archive_path)
        for entry in (os.listdir(local_dir) if os.path.isdir(local_dir) else []):
            entry_path = os.path.join(local_dir, entry)
            if os.path.isdir(entry_path) and entry.startswith("ncbi-blast-"):
                shutil.rmtree(entry_path, ignore_errors=True)

    if wanted.issubset(extracted):
        print(f"✅ Installed blastn, blastdbcmd, and makeblastdb into ./{local_dir}/. BLAST search is ready.")
    else:
        print(f"⚠️ The archive did not contain: {', '.join(wanted - extracted)}. Install manually, following the SETUP note above.")

# The prebuilt search database: a set of files sharing one name prefix (<name>.nin, .nhr, .nsq,
# ...). Defaults to a "blast_db" folder next to this notebook. Build it once from the reference
# transcriptome. See the SETUP note above. This whole BLAST section is optional. The rest of
# the dashboard works fine without it.
BLAST_DB_PATH = os.path.join("blast_db", "pare_transcriptome")

_COMMON_BLAST_INSTALL_DIRS = [
    "/opt/homebrew/bin", "/usr/local/bin", "/opt/local/bin",
    "/usr/local/ncbi/blast/bin", "/opt/ncbi-blast/bin",
    os.path.expanduser("~/miniconda3/bin"), os.path.expanduser("~/anaconda3/bin"),
    os.path.expanduser("~/miniconda3/envs/blast/bin"), os.path.expanduser("~/anaconda3/envs/blast/bin"),
    r"C:\Program Files\NCBI\blast-2.17.0+\bin", r"C:\Program Files\NCBI\blast\bin",
]


def _find_blast_tool(tool_name):
    """Locate a BLAST+ executable: honour BLAST_BIN_DIR if it is set, then check this notebook's
    own ./blast_bin/ (in case install_blast_locally() put it there, this run or a previous one),
    then the system PATH, then a short list of common install locations that a Jupyter kernel's
    PATH often does not include even when BLAST+ is genuinely installed (for example, a kernel
    launched from a GUI application frequently does not inherit PATH entries added only in a
    shell startup file such as .zshrc). Returns the full path or None. shutil.which adds the
    right extension (.exe) on Windows automatically."""
    if BLAST_BIN_DIR:
        found = shutil.which(tool_name, path=BLAST_BIN_DIR)
        if found:
            return found
    found = shutil.which(tool_name, path="blast_bin")
    if found:
        return found
    found = shutil.which(tool_name)
    if found:
        return found
    for directory in _COMMON_BLAST_INSTALL_DIRS:
        found = shutil.which(tool_name, path=directory)
        if found:
            return found
    return None


def _offer_blast_install_button():
    """Shown once per session, only if BLAST+ is not already found anywhere (PATH,
    BLAST_BIN_DIR, or a previous ./blast_bin/ installation). Nothing is downloaded unless the
    button below is clicked."""
    if _find_blast_tool("blastn") is not None:
        return
    import ipywidgets as widgets
    from IPython.display import display

    install_button = widgets.Button(
        description="⬇️ Install BLAST+ for me (NCBI, ~150-250MB, one-time)",
        button_style="info", layout=widgets.Layout(width="420px", height="34px"),
    )
    install_output = widgets.Output()

    def _on_click(b):
        install_button.disabled = True
        with install_output:
            install_blast_locally()

    install_button.on_click(_on_click)
    print("ℹ️ BLAST+ is not installed on this machine. The sequence search section below requires")
    print("   it. Click below to download and install it automatically, or follow the SETUP note")
    print("   above this cell to install it manually.")
    display(widgets.VBox([install_button, install_output]))

_offer_blast_install_button()

# Cache of the most recent search, so the export button can act on whatever was last searched
_CURRENT_BLAST_QUERY = None
_CURRENT_BLAST_HITS = None

def _to_fasta(raw_text):
    """Wrap raw pasted text into a single-record FASTA query. Accepts either bare sequence
    (any whitespace/line breaks) or an already-FASTA-formatted block with its own header."""
    raw_text = raw_text.strip()
    if raw_text.startswith(">"):
        return raw_text
    seq = "".join(raw_text.split())
    return f">user_query\n{seq}"

def run_blast_search(raw_sequence, max_hits=10):
    """BLASTs a pasted nucleotide sequence against Pare's transcriptome and returns a results
    DataFrame with one row per hit, including the PladumEVm... gene ID mapped for use in the
    dashboard above (the isoform suffix like 't2' is stripped since expression data is per-gene,
    not per-isoform)."""
    blastn_exe = _find_blast_tool("blastn")
    if blastn_exe is None:
        print("❌ BLAST+ not found on your system. Install it (see the SETUP note above) or set")
        print("   BLAST_BIN_DIR at the top of this cell. The rest of the dashboard works without BLAST.")
        return None
    if not os.path.isfile(BLAST_DB_PATH + ".nin"):
        print(f"❌ BLAST database not found at {BLAST_DB_PATH}. See the setup note above this cell.")
        return None

    fasta_query = _to_fasta(raw_sequence)
    seq_only = "".join(line for line in fasta_query.splitlines() if not line.startswith(">"))
    if len(seq_only) < 15:
        print("❌ Query sequence looks too short (or empty) to BLAST. Paste at least ~15 bases.")
        return None

    with tempfile.NamedTemporaryFile(mode="w", suffix=".fasta", delete=False) as f:
        f.write(fasta_query)
        query_path = f.name

    try:
        result = subprocess.run(
            [blastn_exe, "-query", query_path, "-db", BLAST_DB_PATH,
             "-outfmt", "6 sseqid pident length evalue bitscore stitle",
             "-max_target_seqs", str(max_hits)],
            capture_output=True, text=True, timeout=60
        )
    finally:
        os.remove(query_path)

    if result.returncode != 0:
        print("❌ blastn reported an error:")
        print(result.stderr.strip())
        return None

    lines = [l for l in result.stdout.strip().splitlines() if l.strip()]
    if not lines:
        print("🛑 No BLAST hits found for this sequence in Pare's transcriptome.")
        return None

    rows = []
    for line in lines:
        sseqid, pident, length, evalue, bitscore, stitle = line.split("\t", 5)
        m = re.match(r"(PladumEVm\d+)t\d+", sseqid)
        gene_id = m.group(1) if m else None
        rows.append({
            "Gene_ID": gene_id,
            "In_Dashboard": bool(gene_id) and gene_id in df_mean.index,
            "Transcript_Hit": sseqid,
            "%Identity": float(pident),
            "Alignment_Length": int(length),
            "E-value": float(evalue),
            "Bit_Score": float(bitscore),
        })

    df_hits = pd.DataFrame(rows).sort_values("Bit_Score", ascending=False).reset_index(drop=True)
    return df_hits

def get_sequence_from_db(seq_id):
    """Retrieves a sequence's full FASTA record directly from the BLAST database via blastdbcmd,
    working entirely from the compiled database, without needing the original transcriptome
    FASTA on disk."""
    blastdbcmd_exe = _find_blast_tool("blastdbcmd")
    if blastdbcmd_exe is None:
        print("❌ BLAST+ not found on your system. Install it (see the SETUP note above) or set")
        print("   BLAST_BIN_DIR at the top of this cell. The rest of the dashboard works without BLAST.")
        return None
    result = subprocess.run(
        [blastdbcmd_exe, "-db", BLAST_DB_PATH, "-entry", seq_id],
        capture_output=True, text=True, timeout=30
    )
    if result.returncode != 0:
        return None
    return result.stdout.strip()

def export_blast_hits(query_sequence, df_hits, save_dir="exported_results"):
    """Saves the current BLAST hit table (CSV) and the matched sequences, retrieved straight
    from the database, as a multi-FASTA file (also including the original query sequence for
    reference) to save_dir."""
    if df_hits is None or df_hits.empty:
        print("❌ Nothing to export. Run a BLAST search with results first!")
        return

    os.makedirs(save_dir, exist_ok=True)
    df_hits.to_csv(os.path.join(save_dir, "blast_hits.csv"), index=False)
    print(f"💾 [SAVED] BLAST hits table exported to '{save_dir}/blast_hits.csv'")

    fasta_records = [_to_fasta(query_sequence).replace(">user_query", ">query_sequence", 1)]
    for seq_id in df_hits["Transcript_Hit"]:
        record = get_sequence_from_db(seq_id)
        if record:
            fasta_records.append(record)

    with open(os.path.join(save_dir, "blast_hit_sequences.fasta"), "w") as f:
        f.write("\n".join(fasta_records) + "\n")
    print(f"💾 [SAVED] Query + {len(fasta_records) - 1} hit sequence(s) exported to '{save_dir}/blast_hit_sequences.fasta'")

In [ ]:
import ipywidgets as widgets
from datetime import datetime
from IPython.display import display, clear_output

def _make_browse_handler(target_box):
    """Opens a native OS folder picker so the user can select an existing folder directly,
    instead of typing or remembering a path. If the dialog is cancelled or unavailable, the text
    box is left unchanged. A different native dialog is used for each operating system: macOS
    uses AppleScript's "choose folder" command, run via osascript in a separate process. Windows
    uses tkinter. Linux tries zenity or kdialog first, falling back to tkinter if neither is
    present. tkinter is deliberately avoided on macOS, because opening a tkinter dialog inside a
    live Jupyter kernel process reliably crashes or hangs the kernel there (Tk's window server
    integration conflicts with the kernel's own Cocoa and asyncio event loop). Running the dialog
    as a separate process, as is done for macOS and Linux, avoids this problem entirely. tkinter
    remains stable when run in process on Windows."""
    def _browse(b):
        import platform
        import subprocess
        import shutil as _shutil

        system = platform.system()
        folder = None
        try:
            if system == "Darwin":
                result = subprocess.run(
                    ["osascript", "-e",
                     'POSIX path of (choose folder with prompt "Choose a folder to save results into")'],
                    capture_output=True, text=True, timeout=120
                )
                if result.returncode == 0:
                    folder = result.stdout.strip()
            elif system == "Windows":
                import tkinter as tk
                from tkinter import filedialog
                root = tk.Tk()
                root.withdraw()
                root.attributes("-topmost", True)
                folder = filedialog.askdirectory(title="Choose a folder to save results into")
                root.destroy()
            else:
                for cmd in (["zenity", "--file-selection", "--directory",
                             "--title=Choose a folder to save results into"],
                            ["kdialog", "--getexistingdirectory"]):
                    if _shutil.which(cmd[0]):
                        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
                        if result.returncode == 0:
                            folder = result.stdout.strip()
                        break
                else:
                    import tkinter as tk
                    from tkinter import filedialog
                    root = tk.Tk()
                    root.withdraw()
                    folder = filedialog.askdirectory(title="Choose a folder to save results into")
                    root.destroy()
        except Exception as e:
            print(f"⚠️ Could not open a folder picker ({e}). Type the path into the box above instead.")
            folder = None

        if folder:
            target_box.value = folder
    return _browse

blast_query_box = widgets.Textarea(
    placeholder="Paste a raw nucleotide sequence here (with or without a '>' FASTA header)...",
    description="Sequence:",
    layout=widgets.Layout(width="600px", height="120px")
)
blast_button = widgets.Button(description="🔬 Run BLAST Search", button_style="warning", layout=widgets.Layout(width="270px", height="40px"))
blast_export_button = widgets.Button(description="💾 Export Hit Sequences", button_style="info", layout=widgets.Layout(width="270px", height="40px"))
blast_save_location_box = widgets.Text(placeholder="exported_results (default if left blank)", description="Save to:", layout=widgets.Layout(width="360px"))
blast_browse_button = widgets.Button(description="📁 Browse...", layout=widgets.Layout(width="120px"))
blast_browse_button.on_click(_make_browse_handler(blast_save_location_box))
blast_output = widgets.Output()

def on_blast_clicked(b):
    global _CURRENT_BLAST_QUERY, _CURRENT_BLAST_HITS
    with blast_output:
        clear_output(wait=True)
        raw_sequence = blast_query_box.value.strip()
        if not raw_sequence:
            print("❌ Paste a sequence first!")
            return

        print("🔬 Running BLAST search against Paré's transcriptome...")
        df_hits = run_blast_search(raw_sequence)
        _CURRENT_BLAST_QUERY = raw_sequence
        _CURRENT_BLAST_HITS = df_hits
        if df_hits is None:
            return

        print(f"\n✅ {len(df_hits)} hit(s) found (best match first):")
        display(df_hits)

        best_ids = df_hits.loc[df_hits["In_Dashboard"], "Gene_ID"].dropna().unique().tolist()
        if best_ids:
            print(f"\n📋 Copy into the 'Paste Box' above to view expression profiles: {', '.join(best_ids)}")
        else:
            print("\n⚠️ None of the matched transcripts have a corresponding entry in the expression matrix above.")

def on_blast_export_clicked(b):
    with blast_output:
        clear_output(wait=True)
        _base_dir = blast_save_location_box.value.strip() or "exported_results"
        save_dir = os.path.join(_base_dir, datetime.now().strftime("search_%Y-%m-%d_%H-%M-%S"))
        print(f"📁 Saving this BLAST search to its own new folder: {save_dir}/")
        export_blast_hits(_CURRENT_BLAST_QUERY, _CURRENT_BLAST_HITS, save_dir=save_dir)

blast_button.on_click(on_blast_clicked)
blast_export_button.on_click(on_blast_export_clicked)

print("🔬 BLAST Sequence Search Ready:")
display(widgets.VBox([
    widgets.HTML("<i>Paste a sequence of interest below to find its matching transcript ID(s) in this dataset.</i>"),
    blast_query_box,
    widgets.HBox([blast_save_location_box, blast_browse_button]),
    widgets.HBox([blast_button, blast_export_button]),
    blast_output
]))

In [ ]:
import re
import numpy as np
from IPython.display import display, HTML

# Global tracking variables so the export button knows what is currently active
_CURRENT_GENES = []
_CURRENT_STAGES = []

def _safe_filename(gene_id):
    """Strips characters Windows forbids in filenames, so no gene ID can ever crash plt.savefig."""
    return re.sub(r'[\\/:*?"<>|]', "_", gene_id)

def _bar_palette(cmap, n):
    """Builds the palette used for the per-condition bars. The default keeps the original 'mako'
    look (independent of the heatmap's red-blue default) since bar color here is purely
    decorative, not tied to a z-score axis. Non-default (colorblind) choices reuse the same
    diverging colormap picked for the heatmap, sampled from its saturated half only. Diverging
    colormaps (PuOr, BrBG) have a washed-out near-white color at their exact center, which would
    otherwise make a bar look broken/invisible if landed on."""
    if cmap == "RdBu_r":
        return "mako"
    cmap_obj = plt.get_cmap(cmap)
    if cmap in ("PuOr", "BrBG"):
        positions = np.linspace(0.60, 0.95, max(n, 1))
    else:
        positions = np.linspace(0.15, 0.85, max(n, 1))
    return [cmap_obj(p) for p in positions]

def _lookup_annotation(gene_id):
    """Check the curated marker list first, then fall back to the Trinotate BLASTX description."""
    rows = []
    marker_hit = df_markers[df_markers["transcriptome_id"] == gene_id]
    for _, r in marker_hit.iterrows():
        category = r["category"]
        rows.append({"Gene_ID": gene_id, "Source": f"Curated marker ({category})",
                      "Symbol_or_Description": r["gene_symbol"]})

    if gene_id in df_annotation.index:
        desc = df_annotation.loc[gene_id, "blastx_description"]
        if isinstance(desc, str):
            rows.append({"Gene_ID": gene_id, "Source": "Trinotate BLASTX", "Symbol_or_Description": desc})

    return rows

def _deg_status_table(gene_id):
    """Sequential-comparison DEG status: prefer the paper's own precomputed calls for curated marker
    genes, otherwise derive Up/Down/Not DEG from presence in the DEG results table."""
    marker_row = df_markers[df_markers["transcriptome_id"] == gene_id]

    rows = []
    for comp_key, comp_label in SEQUENTIAL_COMPARISONS:
        if not marker_row.empty:
            deg_col = "DEG_" + comp_key.replace("vs", "_vs_")
            status = marker_row.iloc[0][deg_col]
        else:
            logfc = _deg_lookup.get((comp_key, gene_id))
            if logfc is None:
                status = "Not DEG"
            else:
                status = "Up regulated" if logfc > 0 else "Down regulated"
        rows.append({"Comparison": comp_label, "Status": status})

    return pd.DataFrame(rows)

def plot_multi_genes(gene_list, stages_chosen=None):
    """Resolves the gene/stage selection and stores it in _CURRENT_GENES/_CURRENT_STAGES.
    Call _render_bar_charts()/_print_annotations() afterward to actually display output. This is
    split out so the caller controls display order (heatmap, then plots, then annotations)."""
    global _CURRENT_GENES, _CURRENT_STAGES

    stages = [s for s in STAGE_ORDER if not stages_chosen or s in stages_chosen]
    if not stages:
        stages = STAGE_ORDER

    _CURRENT_GENES = [g.strip() for g in gene_list if g.strip() in df_mean.index]
    _CURRENT_STAGES = stages

def _render_bar_charts(cmap="RdBu_r"):
    for gene_id in _CURRENT_GENES:
        gene_data = df_mean.loc[gene_id][_CURRENT_STAGES]

        fig, ax = plt.subplots(figsize=(max(3, len(gene_data.index) * 1.3), 4.5))
        sns.barplot(
            x=gene_data.index,
            y=gene_data.values.astype(float),
            hue=gene_data.index,
            palette=_bar_palette(cmap, len(gene_data.index)),
            legend=False,
            width=0.6
        )

        plt.title(f"Expression Profile: {gene_id}", fontsize=16, fontweight="bold", pad=12)
        plt.ylabel("TMM-Normalized Expression", fontsize=14, fontweight="bold")
        plt.xlabel("Regeneration Stage (days post-amputation)", fontsize=14, fontweight="bold")
        plt.xticks(rotation=30, ha="right", fontsize=13)
        plt.yticks(fontsize=13)
        sns.despine()
        plt.tight_layout()
        plt.show()

def _print_annotations(gene_list):
    print("\n📋 GENE ANNOTATION & DEG STATUS:")
    for gene_id in gene_list:
        gene_id = gene_id.strip()
        rows = _lookup_annotation(gene_id)
        if rows:
            print(f"\nAnnotations for Search ID: {gene_id}")
            display(pd.DataFrame(rows))
        else:
            print(f"\nAnnotations for Search ID: {gene_id} -> 🛑 [No annotation found]")

        if gene_id in df_mean.index:
            print(f"Sequential DEG status for {gene_id}:")
            display(_deg_status_table(gene_id))

def _keyword_regex(keyword):
    """Left-word-boundary pattern: the keyword must start at a word boundary (so a short keyword
    cannot silently match as a fragment buried inside an unrelated word, e.g., "vasa" matching
    "extravasation"), but is still free to be a PREFIX of a longer word (so "kinase" still
    matches "kinases", "kinase-like", etc.)."""
    return r"\b" + re.escape(keyword)

def _search_annotations_by_keyword(keyword):
    """Finds every gene whose curated marker symbol or Trinotate BLASTX description contains the
    given keyword as a word-initial match (see _keyword_regex), returning gene IDs actually
    searchable in this dashboard (i.e. present in df_mean.index)."""
    keyword = keyword.strip()
    if not keyword:
        return []
    pattern = _keyword_regex(keyword)
    matched = set()

    marker_mask = df_markers["gene_symbol"].astype(str).str.contains(pattern, case=False, na=False, regex=True)
    matched.update(df_markers.loc[marker_mask, "transcriptome_id"].astype(str))

    ann_mask = df_annotation["blastx_description"].astype(str).str.contains(pattern, case=False, na=False, regex=True)
    matched.update(df_annotation.index[ann_mask].astype(str))

    matched_ids = [g for g in matched if g in df_mean.index]
    return sorted(matched_ids)

# --- SEPARATE EXPORT EXECUTION FUNCTION ---
def export_cached_results(save_dir="exported_results", cmap="RdBu_r"):
    if not _CURRENT_GENES:
        print("❌ Nothing available to save. Please run a successful query search first!")
        return

    os.makedirs(save_dir, exist_ok=True)

    # 1. Export current annotations + DEG status to CSV
    all_rows = []
    for gene_id in _CURRENT_GENES:
        ann_rows = _lookup_annotation(gene_id)
        deg_table = _deg_status_table(gene_id)
        if ann_rows:
            for r in ann_rows:
                merged = dict(r)
                for _, deg_row in deg_table.iterrows():
                    merged[deg_row["Comparison"]] = deg_row["Status"]
                all_rows.append(merged)
        else:
            merged = {"Gene_ID": gene_id, "Source": "None", "Symbol_or_Description": ""}
            for _, deg_row in deg_table.iterrows():
                merged[deg_row["Comparison"]] = deg_row["Status"]
            all_rows.append(merged)

    if all_rows:
        pd.DataFrame(all_rows).to_csv(os.path.join(save_dir, "selected_gene_annotations.csv"), index=False)
        print(f"💾 [SAVED] Annotation + DEG status CSV created inside '{save_dir}/' folder.")

    # 2. Export current graphs to vector PDFs
    for gene_id in _CURRENT_GENES:
        gene_data = df_mean.loc[gene_id][_CURRENT_STAGES]

        fig, ax = plt.subplots(figsize=(max(3, len(gene_data.index) * 1.3), 4.5))
        sns.barplot(x=gene_data.index, y=gene_data.values.astype(float), hue=gene_data.index,
                    palette=_bar_palette(cmap, len(gene_data.index)), legend=False, width=0.6)
        plt.title(f"Expression Profile: {gene_id}", fontsize=16, fontweight="bold", pad=12)
        plt.ylabel("TMM-Normalized Expression", fontsize=14, fontweight="bold")
        plt.xlabel("Regeneration Stage (days post-amputation)", fontsize=14, fontweight="bold")
        plt.xticks(rotation=30, ha="right", fontsize=13)
        plt.yticks(fontsize=13)
        sns.despine()
        plt.tight_layout()

        plt.savefig(os.path.join(save_dir, f"expression_profile_{_safe_filename(gene_id)}.pdf"), format="pdf", dpi=300, bbox_inches="tight")
        plt.close()

    print(f"💾 [SAVED] Vector PDF plots for {_CURRENT_GENES} generated inside '{save_dir}/' folder.")

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage

# --- Multi-gene comparison view: hierarchically-clustered heatmap (Aim E instructions: "generate an
# interactive heatmap..."). Only meaningful with 2+ genes selected, so it is skipped automatically for
# single-gene searches. Both genes (rows) and conditions (columns) are hierarchically clustered so
# genes/conditions with similar profiles end up next to each other.

CMAP_OPTIONS = {
    "Default": "RdBu_r",
    "Purple-Orange (colorblind-safe)": "PuOr",
    "Brown-Teal (colorblind-safe)": "BrBG",
    "Cividis (colorblind-safe, single-hue)": "cividis",
}

def _zscore_matrix(gene_ids, cols, matrix):
    sub = matrix.loc[gene_ids, cols].astype(float)
    # Row z-score so genes with very different absolute expression levels are still
    # visually comparable on the same color scale.
    # NOTE: std() over a single selected column is undefined (NaN), which silently makes the
    # whole z-scored row NaN even though the raw data itself has real values. This is why
    # validity has to be checked on z (the thing actually plotted), not on the raw matrix.
    row_std = sub.std(axis=1).replace(0, 1)
    z = sub.sub(sub.mean(axis=1), axis=0).div(row_std, axis=0)
    return sub, z

def _render_heatmap(gene_ids, cols, matrix, value_label="Expression", cmap="RdBu_r"):
    sub, z = _zscore_matrix(gene_ids, cols, matrix)

    # Drop any gene/condition with zero valid data among the current selection. Otherwise a
    # fully-blank row or column (common when only some picked genes have data for some of the
    # picked conditions) pads the heatmap out with wasted, confusing empty space.
    z = z.loc[z.notna().any(axis=1), z.notna().any(axis=0)]
    gene_ids = list(z.index)
    cols = list(z.columns)

    # Hierarchical clustering (genes and conditions): reorders rows/columns so similar profiles sit
    # next to each other, shown as dendrograms along the top/left. NaN cells are filled with 0 only
    # for the clustering-distance calculation (scipy's linkage cannot handle NaN). The actual displayed
    # cell still renders blank via mask=, so a gene's real missing data is never disguised as a zero.
    z_for_clustering = z.fillna(0)
    row_linkage = linkage(z_for_clustering, method="average") if len(gene_ids) > 1 else None
    col_linkage = linkage(z_for_clustering.T, method="average") if len(cols) > 1 else None

    g = sns.clustermap(
        z, cmap=cmap, center=0, mask=z.isna(),
        row_linkage=row_linkage, col_linkage=col_linkage,
        row_cluster=len(gene_ids) > 1, col_cluster=len(cols) > 1,
        linewidths=0.5, annot=False,
        figsize=(max(6, len(cols) * 0.9) + 2.2, max(4, len(gene_ids) * 1.3 + 2)),
        cbar_pos=(0.90, 0.86, 0.025, 0.10), dendrogram_ratio=(0.15, 0.15),
    )

    # sns.clustermap lets the heatmap/col-dendrogram extend almost to the figure's right edge
    # regardless of cbar_pos (the colorbar floats independently, on top), so it silently overlaps
    # the heatmap's top-right corner. Directly shrink the heatmap + column-dendrogram axes to end
    # before the colorbar's left edge, NOT via fig.subplots_adjust(), which was found (by direct
    # testing) to trigger an internal seaborn recompute that snaps cbar_pos back to its own default
    # top-left position instead of respecting the value passed in. The colorbar is also placed
    # above the heatmap (not beside its top rows) so it never collides with row labels either.
    cbar_left, margin = 0.90, 0.02
    for ax in (g.ax_heatmap, g.ax_col_dendrogram):
        pos = ax.get_position()
        ax.set_position([pos.x0, pos.y0, (cbar_left - margin) - pos.x0, pos.height])

    g.ax_heatmap.set_xlabel("Regeneration Stage (days post-amputation)", fontsize=16, fontweight="bold")
    g.ax_heatmap.set_ylabel("")
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=30, ha="right", fontsize=14)
    plt.setp(g.ax_heatmap.get_yticklabels(), rotation=-30, ha="left", va="center", fontsize=13)
    g.ax_cbar.set_title("Row\nz-score", fontsize=11, fontweight="bold")
    g.ax_cbar.tick_params(labelsize=8)
    g.fig.suptitle(f"Expression Heatmap ({value_label}, hierarchically clustered)",
                   fontsize=17, fontweight="bold", x=0.42, ha="center", y=1.08)
    return g.fig

def _has_any_valid_data(gene_ids, cols, matrix):
    cols = list(dict.fromkeys(cols))  # defensive: guard against any duplicate column labels
    _, z = _zscore_matrix(gene_ids, cols, matrix)
    return bool(z.notna().to_numpy().sum())

def render_multi_gene_views(gene_ids, cols, matrix, value_label="TMM-Normalized Expression", cmap="RdBu_r"):
    """Call after plot_multi_genes(): shows the clustered heatmap when 2+ genes are selected."""
    if len(gene_ids) < 2:
        return
    cols = list(dict.fromkeys(cols))
    if not _has_any_valid_data(gene_ids, cols, matrix):
        print("\n⚠️ No valid data for the selected genes across the chosen conditions. Heatmap skipped.")
        return
    print("\n📊 MULTI-GENE COMPARISON HEATMAP:")
    _render_heatmap(gene_ids, cols, matrix, value_label, cmap)
    plt.show()

def save_multi_gene_views(gene_ids, cols, matrix, value_label="TMM-Normalized Expression", cmap="RdBu_r", save_dir="exported_results"):
    """Call after export_cached_results(): saves the heatmap as a PDF when 2+ genes are selected."""
    if len(gene_ids) < 2:
        return
    cols = list(dict.fromkeys(cols))
    if not _has_any_valid_data(gene_ids, cols, matrix):
        print("\n⚠️ No valid data for the selected genes across the chosen conditions. Heatmap skipped.")
        return
    os.makedirs(save_dir, exist_ok=True)
    fig = _render_heatmap(gene_ids, cols, matrix, value_label, cmap)
    fig.savefig(os.path.join(save_dir, "multi_gene_heatmap.pdf"), format="pdf", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"💾 [SAVED] Heatmap PDF generated inside '{save_dir}/' folder.")


In [ ]:
import ipywidgets as widgets
from datetime import datetime
from IPython.display import display, clear_output

def _make_browse_handler(target_box):
    """Opens a native OS folder picker so the user can select an existing folder directly,
    instead of typing or remembering a path. If the dialog is cancelled or unavailable, the text
    box is left unchanged. A different native dialog is used for each operating system: macOS
    uses AppleScript's "choose folder" command, run via osascript in a separate process. Windows
    uses tkinter. Linux tries zenity or kdialog first, falling back to tkinter if neither is
    present. tkinter is deliberately avoided on macOS, because opening a tkinter dialog inside a
    live Jupyter kernel process reliably crashes or hangs the kernel there (Tk's window server
    integration conflicts with the kernel's own Cocoa and asyncio event loop). Running the dialog
    as a separate process, as is done for macOS and Linux, avoids this problem entirely. tkinter
    remains stable when run in process on Windows."""
    def _browse(b):
        import platform
        import subprocess
        import shutil as _shutil

        system = platform.system()
        folder = None
        try:
            if system == "Darwin":
                result = subprocess.run(
                    ["osascript", "-e",
                     'POSIX path of (choose folder with prompt "Choose a folder to save results into")'],
                    capture_output=True, text=True, timeout=120
                )
                if result.returncode == 0:
                    folder = result.stdout.strip()
            elif system == "Windows":
                import tkinter as tk
                from tkinter import filedialog
                root = tk.Tk()
                root.withdraw()
                root.attributes("-topmost", True)
                folder = filedialog.askdirectory(title="Choose a folder to save results into")
                root.destroy()
            else:
                for cmd in (["zenity", "--file-selection", "--directory",
                             "--title=Choose a folder to save results into"],
                            ["kdialog", "--getexistingdirectory"]):
                    if _shutil.which(cmd[0]):
                        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
                        if result.returncode == 0:
                            folder = result.stdout.strip()
                        break
                else:
                    import tkinter as tk
                    from tkinter import filedialog
                    root = tk.Tk()
                    root.withdraw()
                    folder = filedialog.askdirectory(title="Choose a folder to save results into")
                    root.destroy()
        except Exception as e:
            print(f"⚠️ Could not open a folder picker ({e}). Type the path into the box above instead.")
            folder = None

        if folder:
            target_box.value = folder
    return _browse

# Glossary Accordion View
key_details = widgets.HTML(value='''
<div style="background-color: #f7f9fa; padding: 12px; border-left: 4px solid #2b5b84; border-radius: 4px; font-family: sans-serif; font-size: 13px; line-height: 1.5;">
    <b>📋 REGENERATION STAGE DICTIONARY (chronological order):</b><br>
    0dpa (immediately after amputation) → 1dpa (wound healing) → 2dpa (blastema formation) →
    3dpa (blastema growth) → 5dpa (growth zone resumes) &nbsp;|&nbsp; NA = non-amputated control<br><br>
    Each bar shown is the <b>mean of that stage's replicates</b> (TMM-normalized expression). dpa = days post-amputation.
</div>
''')
glossary_dropdown = widgets.Accordion(children=[key_details], selected_index=None)
glossary_dropdown.set_title(0, "🔍 Click to view/hide Regeneration Stage Glossary")
glossary_dropdown.layout.width = "670px"

# Inputs
stage_multiselect = widgets.SelectMultiple(
    options=STAGE_ORDER,
    value=tuple(STAGE_ORDER),
    description="Stages:",
    layout=widgets.Layout(width="200px", height="140px")
)
cmap_dropdown = widgets.Dropdown(options=list(CMAP_OPTIONS.keys()), description="Palette:", layout=widgets.Layout(width="420px"))

# Filter List: precompute the sorted gene-ID list and a lowercase mirror ONCE here, rather than
# re-sorting + re-lowering all ~120K gene IDs from df_mean.index inside the keystroke handler on
# every character typed (that used to make typing in this box visibly laggy). The widget is also
# capped to MAX_FILTER_RESULTS entries so a broad search term never asks the browser to render
# tens of thousands of <option> rows at once.
_ALL_GENE_IDS = sorted(df_mean.index.astype(str))
_ALL_GENE_IDS_LOWER = [g.lower() for g in _ALL_GENE_IDS]
MAX_FILTER_RESULTS = 500

search_filter = widgets.Text(placeholder="Type to filter transcripts (e.g. PladumEVm0001...)...", description="Filter List:", layout=widgets.Layout(width="440px"))
gene_multiselect = widgets.SelectMultiple(options=_ALL_GENE_IDS[:MAX_FILTER_RESULTS], description="Select Genes:", layout=widgets.Layout(width="440px", height="140px"))
filter_hint = widgets.HTML(value=f"<i style='font-size:11px;color:#666'>Showing first {MAX_FILTER_RESULTS} of {len(_ALL_GENE_IDS)}. Type to narrow down.</i>")

def update_multiselect_options(change):
    search_term = change["new"].strip().lower()
    if search_term:
        matches = [gid for gid, gid_lower in zip(_ALL_GENE_IDS, _ALL_GENE_IDS_LOWER) if search_term in gid_lower]
    else:
        matches = _ALL_GENE_IDS
    gene_multiselect.options = matches[:MAX_FILTER_RESULTS]
    if len(matches) > MAX_FILTER_RESULTS:
        filter_hint.value = f"<i style='font-size:11px;color:#666'>Showing first {MAX_FILTER_RESULTS} of {len(matches)} matches. Refine your search to narrow further.</i>"
    else:
        filter_hint.value = f"<i style='font-size:11px;color:#666'>{len(matches)} matching entr{'y' if len(matches) == 1 else 'ies'}.</i>"
search_filter.observe(update_multiselect_options, names="value")

multi_gene_input = widgets.Textarea(placeholder="OR paste multiple Gene IDs here (e.g. PladumEVm011161, PladumEVm005914)...", description="Paste Box:", layout=widgets.Layout(width="440px", height="100px"))

# --- Option C: Search by Annotation Keyword. Finds every gene whose marker/BLASTX annotation
# contains a keyword, then runs it through the exact same Heatmap -> Plots -> Annotation pipeline
# as the other two search options, sharing the same output_area below. ---
keyword_search_box = widgets.Text(placeholder="e.g. stem cell, Wnt, neurogenesis...", description="Keyword:", layout=widgets.Layout(width="250px"))
keyword_search_button = widgets.Button(description="🔎 Search Annotations", button_style="warning", layout=widgets.Layout(width="250px", height="30px"))

MAX_KEYWORD_MATCHES = 30

# TWO SEPARATE BUTTON CONTROLS
submit_button = widgets.Button(description="🎯 Generate Visuals (View Only)", button_style="success", layout=widgets.Layout(width="330px", height="40px"))
export_button = widgets.Button(description="💾 Export Active Results (Save CSV/PDF)", button_style="info", layout=widgets.Layout(width="330px", height="40px"))
save_location_box = widgets.Text(placeholder="exported_results (default if left blank)", description="Save to:", layout=widgets.Layout(width="360px"))
browse_button = widgets.Button(description="📁 Browse...", layout=widgets.Layout(width="120px"))
browse_button.on_click(_make_browse_handler(save_location_box))

output_area = widgets.Output()

def on_submit_clicked(b):
    with output_area:
        clear_output(wait=True)
        chosen_genes = list(gene_multiselect.value)
        pasted_genes = [g.strip() for g in multi_gene_input.value.replace(",", " ").split() if g.strip()]
        final_gene_list = sorted(list(set(chosen_genes + pasted_genes)))

        if not final_gene_list:
            print("❌ Input target genes first!")
            return

        cmap = CMAP_OPTIONS[cmap_dropdown.value]
        # Output order: Heatmap -> Plots -> Annotation
        plot_multi_genes(final_gene_list, stages_chosen=list(stage_multiselect.value))
        render_multi_gene_views(_CURRENT_GENES, _CURRENT_STAGES, df_mean, "TMM-Normalized Expression", cmap)
        _render_bar_charts(cmap)
        _print_annotations(final_gene_list)

def on_export_clicked(b):
    with output_area:
        _base_dir = save_location_box.value.strip() or "exported_results"
        save_dir = os.path.join(_base_dir, datetime.now().strftime("search_%Y-%m-%d_%H-%M-%S"))
        print(f"📁 Saving this search to its own new folder: {save_dir}/")
        cmap = CMAP_OPTIONS[cmap_dropdown.value]
        export_cached_results(save_dir=save_dir, cmap=cmap)
        save_multi_gene_views(_CURRENT_GENES, _CURRENT_STAGES, df_mean, "TMM-Normalized Expression", cmap, save_dir=save_dir)

def on_keyword_search_clicked(b):
    with output_area:
        clear_output(wait=True)
        keyword = keyword_search_box.value.strip()
        if not keyword:
            print("❌ Type a keyword first!")
            return

        matched_ids = _search_annotations_by_keyword(keyword)
        if not matched_ids:
            print(f"🛑 No genes found with '{keyword}' in their annotation.")
            return

        print(f"✅ Found {len(matched_ids)} gene(s) matching '{keyword}'.")
        if len(matched_ids) > MAX_KEYWORD_MATCHES:
            print(f"⚠️ Limiting to the first {MAX_KEYWORD_MATCHES} matches to keep the plots manageable.")
            matched_ids = matched_ids[:MAX_KEYWORD_MATCHES]

        cmap = CMAP_OPTIONS[cmap_dropdown.value]
        # Output order: Heatmap -> Plots -> Annotation
        plot_multi_genes(matched_ids, stages_chosen=list(stage_multiselect.value))
        render_multi_gene_views(_CURRENT_GENES, _CURRENT_STAGES, df_mean, "TMM-Normalized Expression", cmap)
        _render_bar_charts(cmap)
        _print_annotations(matched_ids)

submit_button.on_click(on_submit_clicked)
export_button.on_click(on_export_clicked)
keyword_search_button.on_click(on_keyword_search_clicked)

# Layout Setup
print("📊 Regeneration Expression Dual-Control Workspace Ready:")
display(widgets.VBox([
    glossary_dropdown,
    widgets.HTML("<br><b>Step 1: Choose Regeneration Stages</b>"),
    widgets.HBox([stage_multiselect, widgets.VBox([cmap_dropdown])]),
    widgets.HTML("<br><b>Step 2: Choose Target Transcripts</b>"),
    widgets.HBox([
        widgets.VBox([widgets.HTML("<i>Option A: Filter List</i>"), search_filter, gene_multiselect, filter_hint]),
        widgets.VBox([widgets.HTML("<i>Option B: Paste Box</i>"), multi_gene_input]),
        widgets.VBox([widgets.HTML("<i>Option C: Search by Keyword</i>"), keyword_search_box, keyword_search_button]),
    ]),
    widgets.HTML("<br><b>Step 3: Run and/or Save</b>"),
    widgets.HBox([save_location_box, browse_button]),
    widgets.HBox([submit_button, export_button]),
    output_area
]))